In [ ]:
# use echo hiding
import numpy as np
import os
import pickle
import hashlib
import librosa
import soundfile as sf
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives.serialization import (
    Encoding, PrivateFormat, PublicFormat, NoEncryption
)

class EchoHidingWatermarker:
    """Audio watermarking class implementing Echo Hiding technique"""
    
    def __init__(self, watermark_bits=32, delay_zero=2, delay_one=4, alpha=0.5, kernel_size=1024):
        """
        Initialize the Echo Hiding watermarker
        
        Args:
            watermark_bits: Number of bits in the watermark
            delay_zero: Delay (in samples) for embedding a '0' bit
            delay_one: Delay (in samples) for embedding a '1' bit
            alpha: Echo amplitude (controls imperceptibility vs. robustness)
            kernel_size: Size of audio segments for embedding each bit
        """
        self.watermark_bits = watermark_bits
        self.delay_zero = delay_zero
        self.delay_one = delay_one
        self.alpha = alpha
        self.kernel_size = kernel_size
        
        # Generate standard watermark sequence - the same for all files
        np.random.seed(42)  # Fixed seed for reproducibility
        self.standard_watermark = np.random.randint(0, 2, size=watermark_bits).astype(np.int8)
    
    def create_echo_kernel(self, delay, alpha):
        """Create an echo kernel with specific delay and alpha"""
        kernel = np.zeros(self.kernel_size)
        kernel[0] = 1.0  # Original signal
        kernel[delay] = alpha  # Echo
        return kernel
    
    def apply_echo(self, segment, delay, alpha):
        """Apply echo to an audio segment using convolution"""
        echo_kernel = self.create_echo_kernel(delay, alpha)
        # Convolve and take only the valid part to avoid edge effects
        echoed_segment = np.convolve(segment, echo_kernel, mode='full')[:len(segment)]
        return echoed_segment
    
    def embed(self, audio, sr=16000):
        """Embed standard watermark into audio using Echo Hiding technique"""
        watermarked_audio = np.copy(audio)
        
        # Calculate the number of samples needed per bit
        samples_per_bit = self.kernel_size * 2  # We need enough samples for the echo
        
        # Check if audio is long enough
        required_length = self.watermark_bits * samples_per_bit
        if len(audio) < required_length:
            # Pad with zeros if not long enough
            padding = np.zeros(required_length - len(audio))
            watermarked_audio = np.concatenate([watermarked_audio, padding])
        
        # Embed each bit of the watermark
        for i in range(self.watermark_bits):
            bit = self.standard_watermark[i]
            
            # Select the appropriate delay based on the bit value
            delay = self.delay_one if bit == 1 else self.delay_zero
            
            # Get the segment from audio
            start_idx = i * samples_per_bit
            end_idx = start_idx + samples_per_bit
            
            if end_idx <= len(watermarked_audio):
                segment = watermarked_audio[start_idx:end_idx]
                # Apply echo
                echoed_segment = self.apply_echo(segment, delay, self.alpha)
                # Update the audio
                watermarked_audio[start_idx:end_idx] = echoed_segment
        
        # Normalize to prevent clipping
        max_val = np.max(np.abs(watermarked_audio))
        if max_val > 1.0:
            watermarked_audio = watermarked_audio / max_val
        
        return watermarked_audio
    
    def detect(self, audio, sr=16000):
        """Detect standard watermark in audio using Echo Hiding Autocorrelation"""
        samples_per_bit = self.kernel_size * 2
        extracted_bits = np.zeros(self.watermark_bits, dtype=np.int8)
        confidence_values = np.zeros(self.watermark_bits)
        
        # Process each segment
        for i in range(self.watermark_bits):
            start_idx = i * samples_per_bit
            end_idx = start_idx + samples_per_bit
            
            if end_idx > len(audio):
                break  # Avoid out of bounds
            
            segment = audio[start_idx:end_idx]
            
            # Apply cepstrum analysis to detect the echo
            # First, compute the autocorrelation to enhance the echo detection
            autocorr = np.correlate(segment, segment, mode='full')
            center = len(autocorr) // 2
            
            # Extract the positive lag part only (right half of the autocorrelation)
            autocorr = autocorr[center:]
            
            # Look for peaks at the expected delays
            zero_peak = np.mean(autocorr[self.delay_zero-1:self.delay_zero+2])
            one_peak = np.mean(autocorr[self.delay_one-1:self.delay_one+2])
            
            # Decision based on which peak is higher
            bit_value = 1 if one_peak > zero_peak else 0
            extracted_bits[i] = bit_value
            
            # Calculate confidence as the normalized difference between peaks
            confidence = abs(one_peak - zero_peak) / max(one_peak, zero_peak)
            confidence_values[i] = confidence
        
        # Calculate similarity with standard watermark
        similarity = np.sum(extracted_bits == self.standard_watermark) / self.watermark_bits
        
        # Return True if similarity exceeds threshold (70%)
        return similarity > 0.7, similarity


class AudioPreprocessorEchoHiding:
    """Class for preprocessing audio with echo hiding watermarking and signature database"""
    
    def __init__(self, watermark_bits=64, delay_zero=4, delay_one=8, alpha=0.3):
        self.watermarker = EchoHidingWatermarker(
            watermark_bits=watermark_bits,
            delay_zero=delay_zero,
            delay_one=delay_one,
            alpha=alpha
        )
        self.digital_signature = DigitalSignature()  # Reuse the existing DigitalSignature class
        
        # Database to store signatures indexed by content hash
        self.signature_db = {}
    
    def process_directory(self, input_dir, output_dir, sample_rate=16000):
        """Process all audio files in a directory"""
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Process each audio file
        processed_count = 0
        for filename in os.listdir(input_dir):
            if filename.endswith(('.wav', '.mp3', '.flac')):
                try:
                    # Load audio file
                    filepath = os.path.join(input_dir, filename)
                    audio, sr = librosa.load(filepath, sr=sample_rate, mono=True)
                    
                    # Process audio
                    processed_audio = self.process_audio(audio)
                    
                    output_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}_processed_echo.wav")
                    sf.write(output_path, processed_audio, sample_rate)
                    
                    processed_count += 1
                    print(f"Processed {filename} -> {output_path}")
                except Exception as e:
                    print(f"Error processing {filename}: {e}")
        
        # Save signature database
        self.save_signature_db(os.path.join(output_dir, "signature_db_echo.pkl"))
        
        print(f"Processed {processed_count} files.")
        return processed_count
    
    def process_audio(self, audio):
        """Process a single audio file with echo hiding watermarking and digital signature"""
        # 1. Embed echo hiding watermark
        watermarked_audio = self.watermarker.embed(audio)
        
        # 2. Generate content hash for identification
        content_hash = self.digital_signature.generate_content_hash(watermarked_audio)
        
        # 3. Generate digital signature
        signature = self.digital_signature.sign(watermarked_audio)
        
        # 4. Store signature in database
        self.signature_db[content_hash] = signature
        
        return watermarked_audio
    
    def verify_audio(self, audio):
        """Verify audio using watermark detection and signature lookup"""
        # 1. Check if standard watermark is present
        watermark_present, similarity = self.watermarker.detect(audio)
        
        if not watermark_present:
            print("Watermark not detected - Audio classified as SPOOFED")
            return False, 0.0, "No watermark"
        
        # 2. Generate content hash
        content_hash = self.digital_signature.generate_content_hash(audio)
        
        # 3. Look up signature in database
        if content_hash in self.signature_db:
            # 4. Verify signature
            signature_valid = self.digital_signature.verify(
                audio, self.signature_db[content_hash]
            )
            
            if signature_valid:
                print(f"Audio verified successfully (watermark similarity: {similarity:.2f})")
                return True, similarity, "Verified"
            else:
                print(f"Digital signature verification failed (watermark similarity: {similarity:.2f})")
                return False, similarity, "Invalid signature"
        else:
            print(f"Content hash not found in signature database (watermark similarity: {similarity:.2f})")
            return False, similarity, "Unknown content"
    
    def save_signature_db(self, filename):
        """Save the signature database"""
        with open(filename, 'wb') as f:
            pickle.dump({
                'signature_db': self.signature_db,
                'watermark_bits': self.watermarker.watermark_bits,
                'delay_zero': self.watermarker.delay_zero,
                'delay_one': self.watermarker.delay_one,
                'alpha': self.watermarker.alpha,
                'standard_watermark': self.watermarker.standard_watermark,
                'keys': self.digital_signature.export_keys()
            }, f)
        print(f"Signature database saved to {filename}")
    
    def load_signature_db(self, filename):
        """Load the signature database"""
        with open(filename, 'rb') as f:
            data = pickle.load(f)
            self.signature_db = data['signature_db']
            self.watermarker.watermark_bits = data['watermark_bits']
            self.watermarker.delay_zero = data['delay_zero']
            self.watermarker.delay_one = data['delay_one']
            self.watermarker.alpha = data['alpha'] 
            self.watermarker.standard_watermark = data['standard_watermark']
            
        print(f"Signature database loaded from {filename}")


class AudioSpoofingDetectionSystemEchoHiding:
    """Audio spoofing detection system that combines echo hiding watermarking, 
    signature database, and ML-based detection"""
    
    def __init__(self, watermark_bits=64, delay_zero=4, delay_one=8, alpha=0.3, model_type='svm',
                use_mfcc=True, use_gtcc=False, use_spectral=False):
        # Initialize components
        self.preprocessor = AudioPreprocessorEchoHiding(
            watermark_bits=watermark_bits, 
            delay_zero=delay_zero,
            delay_one=delay_one,
            alpha=alpha
        )
        # Reuse the existing feature extractor and ML model
        self.feature_extractor = FeatureExtractor(
            use_mfcc=use_mfcc,
            use_gtcc=use_gtcc,
            use_spectral=use_spectral
        )
        self.ml_model = SpoofingDetectionModel(model_type=model_type)
        
        # Status flags
        self.is_trained = False
        
        # Feature configuration for reporting
        self.feature_config = {
            'mfcc': use_mfcc,
            'gtcc': use_gtcc,
            'spectral': use_spectral
        }
    
    def train(self, bonafide_dir, spoofed_dir, sample_rate=16000):
        """Train the complete system"""
        print("=== Starting Training Phase with Echo Hiding ===")
        print("1. Processing bonafide audio files (echo hiding watermarking + signatures)...")
        
        # Print feature configuration
        print(f"Feature configuration: MFCC: {self.feature_config['mfcc']}, " 
              f"GTCC: {self.feature_config['gtcc']}, " 
              f"Spectral: {self.feature_config['spectral']}")
        
        # Create a directory for processed bonafide audio
        processed_dir = os.path.join(os.path.dirname(bonafide_dir), "processed_real_echo")
        self.preprocessor.process_directory(bonafide_dir, processed_dir, sample_rate)
        
        print("\n2. Extracting features for ML model...")
        
        # Prepare feature data and labels
        features = []
        labels = []
        
        # Process bonafide audio (labeled as 1)
        print("  Extracting features from bonafide audio...")
        for filename in os.listdir(processed_dir):
            if filename.endswith('.wav'):
                try:
                    filepath = os.path.join(processed_dir, filename)
                    audio, sr = librosa.load(filepath, sr=sample_rate, mono=True)
                    
                    # Extract features
                    audio_features = self.feature_extractor.extract_all_features(audio, sr)
                    features.append(audio_features)
                    labels.append(1)  # 1 = real
                except Exception as e:
                    print(f"Error extracting features from {filename}: {e}")
        
        # Process spoofed audio (labeled as 0)
        print("  Extracting features from spoofed audio...")
        for filename in os.listdir(spoofed_dir):
            if filename.endswith(('.wav', '.mp3', '.flac')):
                try:
                    filepath = os.path.join(spoofed_dir, filename)
                    audio, sr = librosa.load(filepath, sr=sample_rate, mono=True)
                    
                    # Extract features
                    audio_features = self.feature_extractor.extract_all_features(audio, sr)
                    features.append(audio_features)
                    labels.append(0)  # 0 = spoofed
                except Exception as e:
                    print(f"Error extracting features from {filename}: {e}")
        
        # Convert to numpy arrays
        X = np.array(features)
        y = np.array(labels)
        
        # Train ML model
        print("\n3. Training ML model...")
        validation_accuracy = self.ml_model.train(X, y)
        
        # Save model
        models_dir = os.path.join(os.path.dirname(bonafide_dir), "models_echo")
        os.makedirs(models_dir, exist_ok=True)
        self.ml_model.save(os.path.join(models_dir, "spoofing_detection_model_echo.pkl"))
        
        self.is_trained = True
        print(f"\n=== Training Complete (Validation Accuracy with Echo Hiding: {validation_accuracy:.4f}) ===")
        
        return validation_accuracy
    
    def detect_spoofing(self, audio_path, sample_rate=16000):
        """Detect if an audio file is spoofed using the improved pipeline with echo hiding"""
        if not self.is_trained and not os.path.exists("models_echo/spoofing_detection_model_echo.pkl"):
            print("Error: System not trained. Please train the system first.")
            return False, 0.0
        
        print("=== Starting Detection with Echo Hiding ===")
        
        # Load audio
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
        
        # Step 1: Verify security features (watermark and signature)
        print("1. Verifying echo hiding watermark and signature...")
        security_passed, similarity, reason = self.preprocessor.verify_audio(audio)
        
        if security_passed:
            print(f"Security verification passed - Audio classified as REAL ({reason})")
            return True, 1.0  # High confidence of being real
        
        # Step 2: If security verification failed, run ML detection as backup
        print("2. Security verification failed - Running ML detection...")
        
        # Extract features
        print("3. Extracting audio features...")
        features = self.feature_extractor.extract_all_features(audio, sr)
        
        # Run ML classification
        print("4. Running ML classification...")
        if not self.is_trained:
            self.ml_model.load("models_echo/spoofing_detection_model_echo.pkl")
            self.is_trained = True
            
        is_real, probabilities = self.ml_model.predict(features)
        
        # Final decision based on ML model (security already failed)
        spoofed_probability = probabilities[0] if not is_real else 1.0 - probabilities[1]
        confidence = (1.0 - spoofed_probability) if is_real else spoofed_probability
        
        result = "REAL" if is_real else "SPOOFED"
        print(f"=== Detection Result: {result} (Confidence: {confidence:.4f}, Security: {reason}) ===")
        
        return is_real, confidence
    
    def load_system(self, signature_db_path, model_path):
        """Load a previously trained system"""
        # Load signature database
        self.preprocessor.load_signature_db(signature_db_path)
        
        # Load ML model
        self.ml_model.load(model_path)
        self.is_trained = True
        
        print("Echo Hiding System loaded successfully")
        
    def save_system(self, output_dir):
        """Save the complete system"""
        os.makedirs(output_dir, exist_ok=True)
        
        # Save signature database
        signature_db_path = os.path.join(output_dir, "signature_db_echo.pkl")
        self.preprocessor.save_signature_db(signature_db_path)
        
        # Save ML model
        model_path = os.path.join(output_dir, "spoofing_detection_model_echo.pkl")
        self.ml_model.save(model_path)
        
        # Save feature configuration
        feature_config_path = os.path.join(output_dir, "feature_config_echo.pkl")
        with open(feature_config_path, 'wb') as f:
            pickle.dump(self.feature_config, f)
        
        print(f"Echo Hiding System saved to {output_dir}")


# Example usage
if __name__ == "__main__":
    # Feature configuration
    USE_MFCC = False
    USE_GTCC = True
    USE_SPECTRAL = False
    
    print("Using Echo Hiding technique with GTCC features for training and testing")
    
    # Initialize the echo hiding system
    echo_system = AudioSpoofingDetectionSystemEchoHiding(
        watermark_bits=64,
        delay_zero=4,          # Shorter delay for embedding '0'
        delay_one=8,           # Longer delay for embedding '1'
        alpha=0.3,             # Echo strength (lower is more imperceptible)
        model_type='svm',
        use_mfcc=USE_MFCC,
        use_gtcc=USE_GTCC,
        use_spectral=USE_SPECTRAL
    )
    
    # Define real and fake subdirectories
    train_real_dir = "./archive/train/real/"
    train_fake_dir = "./archive/train/fake/"
    
    # Training phase
    echo_system.train(
        bonafide_dir=train_real_dir,
        spoofed_dir=train_fake_dir,
        sample_rate=16000
    )
    
    # Save the trained system
    echo_system.save_system("trained_system_echo_svm_gtcc")
    
    # Process evaluation real files with echo hiding watermarks
    print("\n=== Processing Evaluation Real Files with Echo Hiding Watermarks ===")
    
    eval_real_processed_dir = "./archive/eval/processed_real_echo"
    os.makedirs(eval_real_processed_dir, exist_ok=True)
    echo_system.preprocessor.process_directory(
        input_dir="./archive/eval/real",
        output_dir=eval_real_processed_dir,
        sample_rate=16000
    )
    
    # Testing phase
    test_results = []
    
    # Test real samples (using processed echo hiding directory)
    print("\n=== Testing Real Audio Samples with Echo Hiding ===")
    test_real_dir = eval_real_processed_dir  # Use the processed directory
    for filename in os.listdir(test_real_dir):
        if filename.endswith(('.wav', '.mp3', '.flac')):
            audio_path = os.path.join(test_real_dir, filename)
            is_real, confidence = echo_system.detect_spoofing(
                audio_path=audio_path,
                sample_rate=16000
            )
            test_results.append({
                'file': filename,
                'actual': 'REAL',
                'predicted': 'REAL' if is_real else 'SPOOFED',
                'confidence': confidence
            })
            print(f"{filename} is {'REAL' if is_real else 'SPOOFED'} with {confidence*100:.2f}% confidence")
    
    # Test fake samples
    print("\n=== Testing Fake Audio Samples with Echo Hiding ===")
    test_fake_dir = "./archive/eval/fake/"
    for filename in os.listdir(test_fake_dir):
        if filename.endswith(('.wav', '.mp3', '.flac')):
            audio_path = os.path.join(test_fake_dir, filename)
            is_real, confidence = echo_system.detect_spoofing(
                audio_path=audio_path,
                sample_rate=16000
            )
            test_results.append({
                'file': filename,
                'actual': 'FAKE',
                'predicted': 'REAL' if is_real else 'SPOOFED',
                'confidence': confidence
            })
            print(f"{filename} is {'Real' if is_real else 'SPOOFED'} with {confidence*100:.2f}% confidence")
    
    # Calculate overall accuracy
    correct = sum(1 for result in test_results if 
                  (result['actual'] == 'REAL' and result['predicted'] == 'REAL') or
                  (result['actual'] == 'FAKE' and result['predicted'] == 'SPOOFED'))
    accuracy = correct / len(test_results) if test_results else 0
    print(f"\nOverall Testing Accuracy with Echo Hiding: {accuracy*100:.2f}%")
    
    # Save results to CSV
    import csv
    with open('test_results_svm_echo_gtcc.csv', 'w', newline='') as csvfile:
        fieldnames = ['file', 'actual', 'predicted', 'confidence']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for result in test_results:
            writer.writerow(result)
    
    print("Results saved to test_results_svm_echo_hiding_gtcc.csv")
